## Importando a base e padronizando a base

In [1]:
import pandas as pd
import numpy as np
import re
from typing import List, Dict

In [2]:
def carregar_padronizar_base(arquivo: str, config_abas: Dict[str, Dict[str, str]], colunas_alvo: List[str]):
    
    """
    Lê múltiplas abas de um arquivo Excel, padroniza as nomenclaturas das colunas, 
    adiciona o ano de referência (carimbo de tempo) e concatena os dados em 
    um único DataFrame no formato Long.
    
    Parâmetros:
    - arquivo (str): Caminho para o arquivo Excel.
    - config_abas (Dict): Dicionário onde a chave é o ano (nome da aba) e o valor 
                          é outro dicionário de renomeação (De -> Para).
    - colunas_alvo (List): Lista com os nomes padronizados das colunas desejadas.
    
    Retorno:
    - pd.DataFrame: Base pronta para análise.
    """

    dataframes = []

    for ano_str, dic_colunas in config_abas.items():
        print (f"Processando os dados do ano: {ano_str}...")

        #Lê a aba correspondente:
        df_temp = pd.read_excel(arquivo, sheet_name=ano_str)
    
        #Padroniza as colunas:
        df_temp = df_temp.rename(columns=dic_colunas)
    
        #Cria a feature do ano:
        ano = int(re.search(r'\d+', ano_str).group())
        df_temp['Ano_avaliacao'] = ano
    
        #Seleciona apenas as colunas alvos que exixtem no dataframe:
        colunas = [col for col in colunas_alvo if col in df_temp.columns]
        df_temp = df_temp[colunas]
    
        #Armaneza na lista:
        dataframes.append(df_temp)

    print('Empilhando as bases ...')

    #Concatena as bases:
    df_final = pd.concat(dataframes, ignore_index=True)
    print(f"Processamento concluído! Base final: { df_final.shape[0]} linhas e {df_final.shape[1]} colunas.")
    return df_final

In [3]:
arquivo = 'BASE DE DADOS PEDE 2024 - DATATHON.xlsx'

#Configuração paramétrica - fácil manutenção para próximos anos:
config_abas = {
    'PEDE2022': {'Nome':'Nome_aluno', 'Idade 22':'Idade', 'Matem':'Nota_mat', 'Portug':'Nota_port', 'Inglês': 'Nota_ing', 'INDE 22':'INDE', 'Pedra 22':'Pedra_atual',
                 'Instituição de ensino':'Instituicao_ensino', 'Defas':'Defasagem'},
    
    'PEDE2023': {'Nome Anonimizado':'Nome_aluno', 'Mat':'Nota_mat', 'Por':'Nota_port', 'Ing': 'Nota_ing', 'INDE 2023':'INDE', 'Pedra 2023':'Pedra_atual',
                 'Instituição de ensino':'Instituicao_ensino', 'Fase Ideal': 'Fase ideal'},

    'PEDE2024': {'Nome Anonimizado':'Nome_aluno','Mat':'Nota_mat', 'Por':'Nota_port', 'Ing': 'Nota_ing', 'INDE 2024':'INDE', 'Pedra 2024':'Pedra_atual',
                 'Instituição de ensino':'Instituicao_ensino', 'Fase Ideal': 'Fase ideal'}
}

colunas_analise = ['RA', 'Ano_avaliacao', 'Nome_aluno', 'Idade', 'Fase', 'Turma', 'Instituicao_ensino', 'INDE', 'Pedra_atual', 'Pedra 20', 'Pedra 21','IDA', 'IEG', 'IAA', 
                   'IPS', 'IPP', 'IPV',
                   'IAN', 'Nota_mat', 'Nota_port', 'Nota_ing', 'Atingiu PV', 'Defasagem', 'Fase ideal']

In [4]:
#Chamando a função

df = carregar_padronizar_base(arquivo=arquivo, config_abas = config_abas, colunas_alvo=colunas_analise)

Processando os dados do ano: PEDE2022...
Processando os dados do ano: PEDE2023...
Processando os dados do ano: PEDE2024...
Empilhando as bases ...
Processamento concluído! Base final: 3030 linhas e 24 colunas.


In [5]:
df.head()

,RA,Ano_avaliacao,Nome_aluno,Idade,Fase,Turma,Instituicao_ensino,INDE,Pedra_atual,Pedra 20,...,IPS,IPV,IAN,Nota_mat,Nota_port,Nota_ing,Atingiu PV,Defasagem,Fase ideal,IPP
0,RA-1,2022,Aluno-1,19,7,A,Escola Pública,5.783,Quartzo,Ametista,...,5.6,7.278,5.0,2.7,3.5,6.0,Não,-1,Fase 8 (Universitários),NaN
1,RA-2,2022,Aluno-2,17,7,A,Rede Decisão,7.055,Ametista,Ametista,...,6.3,6.778,10.0,6.3,4.5,9.7,Não,0,Fase 7 (3º EM),NaN
2,RA-3,2022,Aluno-3,17,7,A,Rede Decisão,6.591,Ágata,Ametista,...,5.6,7.556,10.0,5.8,4.0,6.9,Não,0,Fase 7 (3º EM),NaN
3,RA-4,2022,Aluno-4,17,7,A,Rede Decisão,5.951,Quartzo,Ametista,...,5.6,5.278,10.0,2.8,3.5,8.7,Não,0,Fase 7 (3º EM),NaN
4,RA-5,2022,Aluno-5,17,7,A,Rede Decisão,7.427,Ametista,Ametista,...,5.6,7.389,10.0,7.0,2.9,5.7,Não,0,Fase 7 (3º EM),NaN


## Verificando nulos

In [6]:
#Criando um dataframe de diagnóstico de valores nulos
resumo_nulos = pd.DataFrame({'Total de nulos': df.isnull().sum(),
                             '% de nulos': (df.isnull().sum() / len(df)) * 100})

#filtrando apenas colunas que têm nulos
nulos_filtrado = resumo_nulos[resumo_nulos['Total de nulos'] > 0].sort_values(by='% de nulos', ascending=False)

#Formatando visual
nulos_filtrado['% de nulos'] = nulos_filtrado['% de nulos'].map('{:.2f}%'.format)

print('----Valores nulos----')
print(f"\n{nulos_filtrado}")

----Valores nulos----

                    Total de nulos % de nulos
Pedra 20                      2276     75.12%
Atingiu PV                    2170     71.62%
Pedra 21                      1969     64.98%
Nota_ing                      1939     63.99%
IPP                           1038     34.26%
Nota_port                      185      6.11%
Nota_mat                       184      6.07%
IDA                            178      5.87%
IPV                            178      5.87%
IPS                            171      5.64%
IAA                            165      5.45%
INDE                           147      4.85%
Pedra_atual                    147      4.85%
IEG                             76      2.51%
Instituicao_ensino               1      0.03%


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RA                  3030 non-null   object 
 1   Ano_avaliacao       3030 non-null   int64  
 2   Nome_aluno          3030 non-null   object 
 3   Idade               3030 non-null   object 
 4   Fase                3030 non-null   object 
 5   Turma               3030 non-null   object 
 6   Instituicao_ensino  3029 non-null   object 
 7   INDE                2883 non-null   object 
 8   Pedra_atual         2883 non-null   object 
 9   Pedra 20            754 non-null    object 
 10  Pedra 21            1061 non-null   object 
 11  IDA                 2852 non-null   float64
 12  IEG                 2954 non-null   float64
 13  IAA                 2865 non-null   float64
 14  IPS                 2859 non-null   float64
 15  IPV                 2852 non-null   float64
 16  IAN   

### Tratando nulos e erros vindo do excel em colunas específicas

In [8]:
def limpando_base(df):
    
    """
    Realiza a higienização da base de dados da Passos Mágicos, convertendo tipos 
    corretamente e tratando valores nulos de forma estratégica para visualização no Tableau.
    
    Parâmetros:
    - df: O DataFrame bruto (após o merge dos anos).
    
    Retorno:
    - DataFrame limpo e tipado.
    """

    print('Iniciando limpeza dos dados...')

    df_clean = df.copy()

    #Conversão para númerico:

    col_numerica = ['INDE']
    for col in col_numerica:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

    #Tratamento Histórico:

    col_historica = ['Pedra 20', 'Pedra 21']
    for col in col_historica:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna('Pré-ingresso')

    #Tratamento cadastrais:

    col_cadastrais = ['Turma', 'Instituicao_ensino', 'Fase', 'Pedra_atual']
    for col in col_cadastrais:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna('Sem registro')

    #Avaliação:

    col_avaliacao = ['Atingiu PV']
    for col in col_avaliacao:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna('Não avaliado')

    print('Limpeza concluida!')
    return df_clean    

In [9]:
df_limpo = limpando_base(df)
df_limpo.info()

Iniciando limpeza dos dados...
Limpeza concluida!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RA                  3030 non-null   object 
 1   Ano_avaliacao       3030 non-null   int64  
 2   Nome_aluno          3030 non-null   object 
 3   Idade               3030 non-null   object 
 4   Fase                3030 non-null   object 
 5   Turma               3030 non-null   object 
 6   Instituicao_ensino  3030 non-null   object 
 7   INDE                2845 non-null   float64
 8   Pedra_atual         3030 non-null   object 
 9   Pedra 20            3030 non-null   object 
 10  Pedra 21            3030 non-null   object 
 11  IDA                 2852 non-null   float64
 12  IEG                 2954 non-null   float64
 13  IAA                 2865 non-null   float64
 14  IPS                 2859 non-null   float64
 15  IPV  

In [10]:
notas_cols = ['Nota_mat', 'Nota_port', 'Nota_ing']  
notas_existentes = [c for c in notas_cols if c in df_limpo.columns]

todas_notas = df_limpo[notas_existentes].notna().all(axis=1)

IDA_calculado = np.where(
    todas_notas,
    (df[notas_existentes].sum(axis=1) / 3).round(2),
    None
)

# Prioriza o IDA original, preenche com calculado onde for nulo
df_limpo['IDA'] = df_limpo['IDA'].combine_first(pd.Series(IDA_calculado, index=df_limpo.index))

print(f'IDA preenchido: {df_limpo["IDA"].notna().sum()}')
print(f'IDA ainda nulo: {df_limpo["IDA"].isna().sum()}')

IDA preenchido: 2852
IDA ainda nulo: 178


In [11]:
df_limpo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RA                  3030 non-null   object 
 1   Ano_avaliacao       3030 non-null   int64  
 2   Nome_aluno          3030 non-null   object 
 3   Idade               3030 non-null   object 
 4   Fase                3030 non-null   object 
 5   Turma               3030 non-null   object 
 6   Instituicao_ensino  3030 non-null   object 
 7   INDE                2845 non-null   float64
 8   Pedra_atual         3030 non-null   object 
 9   Pedra 20            3030 non-null   object 
 10  Pedra 21            3030 non-null   object 
 11  IDA                 2852 non-null   object 
 12  IEG                 2954 non-null   float64
 13  IAA                 2865 non-null   float64
 14  IPS                 2859 non-null   float64
 15  IPV                 2852 non-null   float64
 16  IAN   

## Coluna idade

In [12]:
df_limpo['Idade'].unique()

array([19, 17, 18, 20, 21, 16, 15, 13, 14, 12, 11, 10, 9, 8, 7,
       datetime.datetime(1900, 1, 8, 0, 0),
       datetime.datetime(1900, 1, 7, 0, 0),
       datetime.datetime(1900, 1, 11, 0, 0),
       datetime.datetime(1900, 1, 9, 0, 0),
       datetime.datetime(1900, 1, 10, 0, 0),
       datetime.datetime(1900, 1, 14, 0, 0),
       datetime.datetime(1900, 1, 13, 0, 0),
       datetime.datetime(1900, 1, 12, 0, 0),
       datetime.datetime(1900, 1, 15, 0, 0),
       datetime.datetime(1900, 1, 17, 0, 0),
       datetime.datetime(1900, 1, 16, 0, 0),
       datetime.datetime(1900, 1, 19, 0, 0),
       datetime.datetime(1900, 1, 18, 0, 0), 22,
       datetime.datetime(1900, 1, 20, 0, 0),
       datetime.datetime(1900, 1, 21, 0, 0),
       datetime.datetime(1900, 1, 26, 0, 0), 23, 24, 27, 25], dtype=object)

In [13]:
def corrigir_idade(valor):
    
    """
    Corrige o bug de formatação de data do Excel e converte idades para número inteiro.
    """

    if pd.isna(valor):
        return np.nan

    #Converte para string:
    valor_str = str(valor).strip()

    #Localizando o bug do excel:
    if '1900-'in valor_str:
        try:
            #Converte para data e extrai o dia que é a idade
            idade_real = pd.to_datetime(valor_str).day
            return int(idade_real)
        except:
            pass

    #Convertendo números normais:
    try:
        return int(float(valor_str))
    except ValueError:
        return np.nan

In [14]:
if 'Idade' in df_limpo.columns:
    df_limpo['Idade'] = df_limpo['Idade'].apply(corrigir_idade)

print(df_limpo['Idade'].unique())

[19 17 18 20 21 16 15 13 14 12 11 10  9  8  7 22 26 23 24 27 25]


In [15]:
df_limpo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RA                  3030 non-null   object 
 1   Ano_avaliacao       3030 non-null   int64  
 2   Nome_aluno          3030 non-null   object 
 3   Idade               3030 non-null   int64  
 4   Fase                3030 non-null   object 
 5   Turma               3030 non-null   object 
 6   Instituicao_ensino  3030 non-null   object 
 7   INDE                2845 non-null   float64
 8   Pedra_atual         3030 non-null   object 
 9   Pedra 20            3030 non-null   object 
 10  Pedra 21            3030 non-null   object 
 11  IDA                 2852 non-null   object 
 12  IEG                 2954 non-null   float64
 13  IAA                 2865 non-null   float64
 14  IPS                 2859 non-null   float64
 15  IPV                 2852 non-null   float64
 16  IAN   

## Limpando algumas colunas

In [16]:
df_limpo['Pedra_atual'].unique()

array(['Quartzo', 'Ametista', 'Ágata', 'Topázio', 'Agata', 'Sem registro',
       'INCLUIR'], dtype=object)

In [17]:
df_limpo['Pedra_atual'] = df_limpo['Pedra_atual'].replace({'Agata':'Ágata', 'INCLUIR':'Sem registro'})

In [18]:
df_limpo['Pedra_atual'].unique()

array(['Quartzo', 'Ametista', 'Ágata', 'Topázio', 'Sem registro'],
      dtype=object)

In [19]:
df_limpo['Pedra 20'].unique()

array(['Ametista', 'Pré-ingresso', 'Quartzo', 'Ágata', 'Topázio'],
      dtype=object)

In [20]:
df_limpo['Pedra 21'].unique()

array(['Ametista', 'Ágata', 'Pré-ingresso', 'Topázio', 'Quartzo'],
      dtype=object)

In [21]:
df_limpo['Instituicao_ensino'].unique()

array(['Escola Pública', 'Rede Decisão', 'Escola JP II', 'Pública',
       'Privada', 'Privada - Programa de Apadrinhamento',
       'Privada - Programa de apadrinhamento', 'Concluiu o 3º EM',
       'Nenhuma das opções acima', 'Privada *Parcerias com Bolsa 100%',
       'Privada - Pagamento por *Empresa Parceira', 'Sem registro',
       'Bolsista Universitário *Formado (a)'], dtype=object)

In [22]:
df_limpo['Instituicao_ensino'] = df_limpo['Instituicao_ensino'].replace({'Privada - Programa de apadrinhamento':'Privada - Programa de Apadrinhamento',
                                        'Privada *Parcerias com Bolsa 100%': 'Privada - Parcerias com Bolsa 100%',
                                        'Privada - Pagamento por *Empresa Parceira': 'Privada - Pagamento por Empresa Parceira',
                                        'Bolsista Universitário *Formado (a)': 'Bolsista Universitário Formado',
                                        'Pública': 'Escola Pública', 'Privada': 'Escola Privada'})

In [23]:
df_limpo['Instituicao_ensino'].unique()

array(['Escola Pública', 'Rede Decisão', 'Escola JP II', 'Escola Privada',
       'Privada - Programa de Apadrinhamento', 'Concluiu o 3º EM',
       'Nenhuma das opções acima', 'Privada - Parcerias com Bolsa 100%',
       'Privada - Pagamento por Empresa Parceira', 'Sem registro',
       'Bolsista Universitário Formado'], dtype=object)

In [24]:
df_limpo['Fase'].unique()

array([7, 6, 5, 4, 3, 2, 1, 0, 'ALFA', 'FASE 1', 'FASE 2', 'FASE 3',
       'FASE 4', 'FASE 5', 'FASE 6', 'FASE 7', 'FASE 8', '1A', '1B', '1C',
       '1D', '1E', '1G', '1H', '1J', '1K', '1L', '1M', '1N', '1P', '1R',
       '2A', '2B', '2C', '2D', '2G', '2H', '2I', '2K', '2L', '2M', '2N',
       '2P', '2R', '2U', '3A', '3B', '3C', '3D', '3F', '3G', '3H', '3I',
       '3K', '3L', '3M', '3N', '3P', '3R', '3U', '4A', '4B', '4C', '4F',
       '4H', '4L', '4M', '4N', '4R', '5A', '5B', '5C', '5D', '5F', '5G',
       '5L', '5M', '5N', '6A', '6L', '7A', '7E', '8A', '8B', '8D', '8E',
       '8F', 9], dtype=object)

In [25]:
def fase(valor):
    
    vlr_str = str(valor).strip().upper()

    if 'ALFA' in vlr_str or vlr_str == '0':
        return 'Alfa'

    #Busca o primeiro digito para outras fases

    dig = re.search(r'\d', vlr_str)

    if dig:
        num = dig.group()
        return f'Fase {num}'

    return vlr_str

df_limpo['Fase'] = df_limpo['Fase'].apply(fase)

In [26]:
df_limpo['Fase'].unique()

array(['Fase 7', 'Fase 6', 'Fase 5', 'Fase 4', 'Fase 3', 'Fase 2',
       'Fase 1', 'Alfa', 'Fase 8', 'Fase 9'], dtype=object)

In [27]:
df_limpo['Turma'].unique()

array(['A', 'D', 'B', 'C', 'F', 'K', 'L', 'H', 'P', 'E', 'G', 'I', 'M',
       'N', 'J', 'Q', 'U', 'R', 'O', 'S', 'T', 'V', 'Y', 'Z',
       'ALFA A - G0/G1', 'ALFA B - G2/G3', 'ALFA C - G0/G1',
       'ALFA D - G2/G3', 'ALFA E - G2/G3', 'ALFA F - G0/G1',
       'ALFA G - G2/G3', 'ALFA H - G0/G1', 'ALFA I - G2/G3',
       'ALFA J - G2/G3', 'ALFA K - G0/G1', 'ALFA L - G2/G3',
       'ALFA M - G0/G1', 'ALFA N - G2/G3', 'ALFA O - G2/G3',
       'ALFA Q - G2/G3', 'ALFA R - G0/G1', 'ALFA S - G2/G3',
       'ALFA T - G2/G3', 'ALFA U - G2/G3', 'ALFA V - G0/G1',
       'ALFA Y - G0/G1', '1A', '1B', '1C', '1D', '1F', '1G', '1H', '1J',
       '1K', '1L', '1M', '1N', '1P', '1R', '2A', '2B', '2C', '2D', '2F',
       '2G', '2H', '2I', '2K', '2L', '2M', '2N', '2P', '2R', '2U', '3A',
       '3B', '3C', '3D', '3F', '3G', '3I', '3K', '3L', '3M', '3N', '3P',
       '4A', '4B', '4C', '4F', '4G', '4H', '4L', '4M', '4N', '5A', '5B',
       '5C', '5F', '5L', '5M', '6A', '6L', '7A', '7E', '8A', '8B', '8D',
 

## Calculo do atingiu IPV

In [28]:
# ============================================================
# CÁLCULO DO PONTO DE VIRADA (PV) — METODOLOGIA PASSOS MÁGICOS (Fonte: Relatório PEDE 2022)
# Critério: nota de corte = média(IPV) + desvio padrão(IPV)
# Validado contra campo original de 2022
# ============================================================


#Calculando a nota de corte por ano

cortes = {}

for ano in df_limpo['Ano_avaliacao'].unique():
    dados = df_limpo[df_limpo['Ano_avaliacao'] == ano]['IPV'].dropna()
    cortes[ano] = dados.mean() + dados.std()

print("Notas de corte por ano:")
for ano, corte in sorted(cortes.items()):
    print(f"   {ano}: {corte:.4f}")

Notas de corte por ano:
   2022: 8.3470
   2023: 8.9733
   2024: 8.4028


In [29]:
def calcular_pv(row):
    if pd.isna(row['IPV']):
        return 'Sem dado'
    return 'Sim'if row['IPV'] >= cortes[row['Ano_avaliacao']] else 'Não'

df_limpo['Atingiu_PV_calculado'] = df_limpo.apply(calcular_pv, axis=1)


print("\nDistribuição por ano:")
print(pd.crosstab(df_limpo['Ano_avaliacao'], df_limpo['Atingiu_PV_calculado']))

# Validação específica 2022 
print("\nValidação 2022 — calculado vs. original:")
print(pd.crosstab(df_limpo[df_limpo['Ano_avaliacao']==2022]['Atingiu PV'], 
                  df_limpo[df_limpo['Ano_avaliacao']==2022]['Atingiu_PV_calculado']))


Distribuição por ano:
Atingiu_PV_calculado  Não  Sem dado  Sim
Ano_avaliacao                           
2022                  747         0  113
2023                  805        76  133
2024                  902       102  152

Validação 2022 — calculado vs. original:
Atingiu_PV_calculado  Não  Sim
Atingiu PV                    
Não                   747    0
Sim                     0  113


## Jornada do aluno - cálculo de evolução

In [30]:
df_vis = df_limpo.copy()

#Ordena a base por aluno e por ano:

df_vis = df_vis.sort_values(by=['RA', 'Ano_avaliacao'])

#Indicadores para cálculo da evolução:

indicadores = ['INDE', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV', 'IAN']

# Marca a primeira ocorrência real de cada aluno na base
primeiro_ano_mask = ~df_vis.duplicated(subset='RA', keep='first')

#Calculo da diferença de um ano para outro:

for ind in indicadores:
    if ind in df_vis.columns:
        df_vis[f"Delta_{ind}"] = df_vis.groupby('RA')[ind].diff()

#Cria a jornada do aluno:

for ind in indicadores:
    delta_col = f"Delta_{ind}"
    evolucao_col = f"Evolucao_{ind}"

    if delta_col in df_vis.columns:
        condicoes = [
            primeiro_ano_mask,
            df_vis[delta_col] > 0,
            df_vis[delta_col] < 0,
            df_vis[delta_col] == 0]
        rotulos = ['Primeiro ano', 'Avanço', 'Recuo', 'Neutra']

    # Se o Delta for NaN, significa que é o primeiro ano do aluno na base (ele não tem ano anterior para comparar)
        df_vis[evolucao_col] = np.select(condicoes, rotulos, default='Sem dados')

## Filtro de persona - clusterização

•	A Aplicação: Nomear os clusters com rótulos semânticos ("Talentos Ansiosos", "Desengajados Críticos", "Estrelas Resilientes"), para que o dashboard deixe de ser um painel de consulta e vire uma ferramenta de diagnóstico rápido. A equipe pedagógica pode filtrar pelos "Desengajados Críticos" e focar os esforços de resgate da semana exatamente neles.

In [31]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [32]:
#Define as colunas que formam a mente e a açao do aluno:

features_cluster = ['IDA', 'IEG', 'IPS', 'IAA']

#Separa apenas os alunos que tem essas 4 notas completas:

df_cluster = df_vis.dropna(subset=features_cluster).copy()

#Padroniza a escala dos dados:

scaler = StandardScaler()
dados_padronizados = scaler.fit_transform(df_cluster[features_cluster])

#Configura e treina e algoritmo (considerando 4 perfis de aluno):
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_cluster['ID_cluster'] = kmeans.fit_predict(dados_padronizados)

#Calcula as médias de cada cluster
medias_cluster = df_cluster.groupby('ID_cluster')[features_cluster].mean()
print('Média das notas por cluster:')
print(f"\n{medias_cluster}")

Média das notas por cluster:

                 IDA       IEG       IPS       IAA
ID_cluster                                        
0            4.25126  6.548833  6.910043  8.481615
1           6.509027  8.707836  3.626407  8.598513
2           7.526434  8.990096  7.256565  8.818590
3           5.913441  7.811727  5.507198  0.039516


Cluster 0:

Os dados: Notas baixíssimas (IDA 4.25) e entrega de lições média-baixa (IEG 6.5). Porém, o psicológico está ok (IPS 6.9) e eles se dão uma nota muito alta na autoavaliação (IAA 8.48).

O Diagnóstico: Eles acham que estão indo muito bem, mas a realidade acadêmica diz o contrário. É o clássico "Dunning-Kruger".

Nome da Persona: 🚨 "Alerta Acadêmico (Desfocados)"

Cluster 1 (O mais crítico para a ONG):

Os dados: Notas na média (IDA 6.5) e alta entrega de lição (IEG 8.7). A autoavaliação também é alta (IAA 8.5). Mas o IPS (Psicossocial) está destruído: 3.62!

O Diagnóstico: São alunos muito esforçados, que tentam cumprir as regras, mas que estão passando por problemas gravíssimos em casa, na família ou no emocional.

Nome da Persona: 🆘 "Esforçados em Risco Emocional" (Esse cluster vai brilhar no Tableau. A assistência social da ONG precisa focar nestes alunos urgentemente antes que evadam).

Cluster 2:

Os dados: Notas altas (IDA 7.5), engajamento altíssimo (IEG 8.9), psicológico saudável (IPS 7.2) e boa autopercepção (IAA 8.8).

O Diagnóstico: É o aluno ideal da Passos Mágicos. Estão colhendo todos os frutos do programa.

Nome da Persona: ⭐ "Alto Desempenho (Estrelas Resilientes)"

Cluster 3:

Os dados: Notas medianas (IDA 5.9), entregam as lições (IEG 7.8) e psicológico mediano (IPS 5.5). Mas o IAA (Autoavaliação) é 0.03!

O Diagnóstico: Alunos que estão "empurrando com a barriga", mas que têm a autoestima completamente apagada ou estão tão desengajados de si mesmos que sequer preenchem/pontuam na autoavaliação.

Nome da Persona: 👻 "Crise de Autopercepção (Invisíveis)"

In [33]:
#Trazer o ID cluster para a base inicial:

df_vis = df_vis.merge(df_cluster[['RA', 'Ano_avaliacao', 'ID_cluster']], on=['RA', 'Ano_avaliacao'], how='left')

#Nomear o cluster:

mapa_persona = {0: 'Alerta acadêmico (desfocados)',
              1: 'Esforçados em risco emocional',
              2: 'Alto desempenho (estrelas resilientes)',
              3: 'Crise de autopercepção (invisíveis)'}

df_vis['Persona_aluno'] = df_vis['ID_cluster'].map(mapa_persona)

df_vis['Persona_aluno'] = df_vis['Persona_aluno'].fillna('Dados incompletos para perfil')

df_vis = df_vis.drop(columns=['ID_cluster'])

In [34]:
df_vis.columns

Index(['RA', 'Ano_avaliacao', 'Nome_aluno', 'Idade', 'Fase', 'Turma',
       'Instituicao_ensino', 'INDE', 'Pedra_atual', 'Pedra 20', 'Pedra 21',
       'IDA', 'IEG', 'IAA', 'IPS', 'IPV', 'IAN', 'Nota_mat', 'Nota_port',
       'Nota_ing', 'Atingiu PV', 'Defasagem', 'Fase ideal', 'IPP',
       'Atingiu_PV_calculado', 'Delta_INDE', 'Delta_IDA', 'Delta_IEG',
       'Delta_IAA', 'Delta_IPS', 'Delta_IPP', 'Delta_IPV', 'Delta_IAN',
       'Evolucao_INDE', 'Evolucao_IDA', 'Evolucao_IEG', 'Evolucao_IAA',
       'Evolucao_IPS', 'Evolucao_IPP', 'Evolucao_IPV', 'Evolucao_IAN',
       'Persona_aluno'],
      dtype='object')

## Salvando o arquivo para montar o dashboard 

In [35]:
df_vis.to_csv('base_dados_passos_magicos.csv', sep=';', index=False, encoding='utf-8-sig')